## Question 3: Natural Language Processing

In [2]:
corpuses = [
"nlp/01-introduction.csv",
"nlp/02-data-exploration.csv",
"nlp/03-decision-trees.csv",
"nlp/04-regression.csv",
"nlp/05-support-vector-machines.csv",
"nlp/06-neural-networks-1.csv",
"nlp/07-neural-networks-2.csv",
"nlp/08-evaluation.csv",
"nlp/09-clustering.csv",
"nlp/10-frequent-itemsets.csv",
]

In [3]:
# Load the dataset
import pandas as pd

masterDF = pd.read_csv(corpuses[0])
masterDF["Lecture Number"] = 1
masterDF["Lecture name"] = corpuses[0].split("nlp/")[1].split(".csv")[0]

for indx, csvFile in enumerate(corpuses[1:]):
    newDF = pd.read_csv(csvFile)
    newDF["Lecture Number"] = indx+2
    newDF["Lecture name"] = csvFile.split("nlp/")[1].split(".csv")[0]
    masterDF = pd.concat([masterDF, newDF],ignore_index=True)

### a)

In [4]:
from collections import Counter
import plotly.express as px

## Use Counter to count individual words
textCol = masterDF["text"]
wordCounter = Counter()

for text in textCol:
    wordCounter.update(text.split(" "))

frequentWordDF = pd.DataFrame(wordCounter.most_common(25), index=range(25))
frequentWordDF = frequentWordDF.rename(columns={0:"word", 1:"count"})

fig = px.histogram(frequentWordDF["count"],title="Histogram of word frequencies", labels={"value": "Word frequency", "count": "Count"})
fig.update_layout(showlegend=False)


The approach is faulty as it can miss counts due to differences in capitalization at the start of a sentence and the presence or absence of punctuation.

### b)

In [5]:
import nltk
from nltk.corpus import stopwords
import string

def tokenize_string(input: str, removeStopWords = False):
    ## Remove punctuation and split the string
    transTable = str.maketrans("", "", string.punctuation)
    cleantext = input.translate(transTable).lower()
    wordList = cleantext.split(" ")

    if removeStopWords:
        ## Create a new list without stopwords
        englStopWords = stopwords.words("english")
        englStopWords.append("") ## Debugging
        finalWordList = [x for x in wordList if x not in englStopWords]

    else:
        finalWordList = [x for x in wordList if x not in [""]]

    ## Tokenize all words and return
    tokenlist = [nltk.word_tokenize(x)[0] for x in finalWordList]

    return tokenlist

## Preprocess the strings
masterTokenlist = []
for text in textCol:
    masterTokenlist.extend(tokenize_string(text,True))

## Use pandas to generate the tokenized word counts
tokenizedValueCountsDF = pd.DataFrame(pd.Series(masterTokenlist).value_counts()).sort_values("count", ascending=False).reset_index()
px.histogram(tokenizedValueCountsDF["count"][:25])

### c)

In [6]:
## Set up the frequency DF and the requested words
requestedWords = ["data", "decision", "predict", "derivative","network", "easy", "database"]
frequencyDF = pd.DataFrame(columns=["word", "lecture", "frequency"])

## Go through all rows in masterDF and collect all tokenized words until the lecture ends
currLecture = 1
lectureWords = []
currLectureWords = []

for rowIndex in masterDF.index:
    lectureNum = masterDF.loc[rowIndex, "Lecture Number"]
    if lectureNum == currLecture:
        currLectureWords.extend(tokenize_string( masterDF.loc[rowIndex, "text"]))
        continue
    else:
        lectureWords.append(currLectureWords)
        currLectureWords = tokenize_string( masterDF.loc[rowIndex, "text"])
        currLecture = lectureNum

## Append the last list
lectureWords.append(currLectureWords)

## Generate the word counts per lecture
for indx, lectureList in enumerate(lectureWords):
    lectureNum = indx+1
    valueCountDict = pd.Series(lectureList).value_counts().to_dict()
    for requestedWord in requestedWords:
        ## Check if word appears in string at all
        if requestedWord in valueCountDict.keys():
            wordCount = valueCountDict[requestedWord]
        else:
            wordCount = 0
        frequencyDF.loc[len(frequencyDF)] = [requestedWord, lectureNum, wordCount]

frequencyDF["lecture"] = frequencyDF["lecture"].astype(str)

px.histogram(frequencyDF, x="word", y="frequency", color="lecture", barmode="stack", labels={"sum of frequency": "Sum of Frequency"})

While data is very evenly distributed across all lectures, other words like database (lecture 10) or derivate (lecture 4 and 7) only appear in a few lectures. This mirrors the way the course handles different topics.

### d)

In [7]:
from nltk.util import ngrams
from nltk.probability import ConditionalFreqDist
import random

random.seed(3213)

## Generate lists of tokenized text segments
tokenizedLists = []

for text in masterDF["text"]:
    stringList = ["<s>"]
    stringList.extend(tokenize_string(text))
    stringList.append("</s>")
    tokenizedLists.append(stringList)

## Define functions
def predictNextWord(input:list):
    ## Ensure context size
    if len(input) > n_value-1:
        input= input[-(n_value-1):]

    if len(input) < n_value-1:
        raise Exception("Input too small")

    ## Get the samples from the cfd
    context = tuple(input)
    samples = cfd[context]

    if len(samples) == 0:
        return None

    ## Choose a random word, weighted by frequency
    wordChoices = sorted(list(samples.keys()))
    wordWeights = [samples[x] for x in wordChoices]
    random.seed(3213)
    result = random.choices(population=wordChoices, weights=wordWeights)[0]

    return result

def generateText(seedText: str, num_words: int):
    ## Tokenize the seed text and ensure its length
    tokenizedSeedText = tokenize_string(seedText)
    if len(tokenizedSeedText) < n_value-1:
        tokenizedSeedText = ["<s>"]*((n_value-1)-len(tokenizedSeedText))+tokenizedSeedText
    currentText = tokenizedSeedText

    ## Append new word to current text, predict the next
    for i in range(num_words):
        newWord = predictNextWord(currentText)
        ## Append only if a word was received
        if newWord is None or newWord == "</s>":
            break
        currentText.append(newWord)


    return " ".join(currentText)

In [8]:
## Generate text for all n Values
for i in [3,4,5,2]:
    n_value = i

    ngramsList = [ngrams(x, n_value) for x in tokenizedLists]
    cfd = ConditionalFreqDist()

    for ngram_generator in ngramsList:
        for ngram in ngram_generator:
            context = tuple(ngram[:n_value-1])
            target = ngram[-1]
            cfd[context][target] += 1

    print("\nText for n = " + str(i)+ ":\n")
    print(generateText("introduction to data", 30))


Text for n = 3:

introduction to data science master and its not shown so again this other one is not possible and so on the part that is possible for specific tasks so the people that invented

Text for n = 4:

introduction to data science so this lecture will be a lecture on regression they will be provided to you so the question earlier was like how to interpret these factors the different rate

Text for n = 5:

<s> introduction to data

Text for n = 2:

introduction to data set on the or regression model prediction model prediction model prediction model prediction model prediction model prediction model prediction model prediction model prediction model prediction model prediction model prediction model


We see that for n=5, no text outside the seed phrase is generated. This is expected as there is no instance of the start of a sequence of words in the dataset being "introduction data science" (potentially with stopwords in between). For n > 5, this is guaranteed to continue indefinately.

### e)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import SnowballStemmer

englishStopwords = list(stopwords.words("english"))
stemmer = SnowballStemmer("english")

def preprocessText(input: str):
    tokenList = tokenize_string(input, removeStopWords=True)
    stemmedOutput = [stemmer.stem(x) for x in tokenList]

    return stemmedOutput

## Create vectorizer and fit to text corpus
lecture_content = masterDF.groupby("Lecture Number")['text'].apply(lambda x: ' '.join(x)).reset_index()
vectorizer_l1 = TfidfVectorizer(analyzer=preprocessText)
tdidf_l1 = vectorizer_l1.fit_transform(lecture_content["text"])

def topKLectures(input:str, k:int):
    ## Vectorize the input and compute the scores
    inputVectorized = vectorizer_l1.transform([input])
    scores = (tdidf_l1.dot(inputVectorized.T).toarray().flatten())
    
    ## Return the top lecture names
    scoreSeries = pd.Series(scores).sort_values(ascending=False)
    topindices = list(scoreSeries.index[:k])
    topLectureNames = [masterDF.drop_duplicates("Lecture name").iloc[x]["Lecture name"] for x in topindices]

    outScores = scoreSeries.values[:k]

    return topLectureNames, outScores

def topMTimestamps(input:str, lectureName:str, m:int):
    ## Vectorize the timestamps
    lectureTimeStamps = masterDF[masterDF["Lecture name"] == lectureName]
    vectorizer_l2 = TfidfVectorizer(analyzer=preprocessText)
    tfidf_l2 = vectorizer_l2.fit_transform(lectureTimeStamps["text"])

    ## Vectorize the input and compute the scores
    inputVectorized = vectorizer_l2.transform([input])
    scores = (tfidf_l2.dot(inputVectorized.T)).toarray().flatten()

    ## Get the indices of the highest scores
    scoreSeries = pd.Series(scores).sort_values(ascending=False)
    topindices = list(scoreSeries.index[:m])

    outScores = scoreSeries.values[:m]

    timestampList = [[lectureTimeStamps.iloc[x]["start"], lectureTimeStamps.iloc[x]["end"]] for x in topindices]

    return timestampList, outScores

def tfidf_search(input, k, m):
    topNames, lectureScores = topKLectures(input, k)
    topTimeStamps = []
    for name in topNames:
        timestamps, scores = topMTimestamps(input, name, m)
        topTimeStamps.append([str(timestamps[x]) + " (" + str(scores[x]) + ")" for x in range(len(timestamps))])

    outDF = pd.DataFrame()
    outDF["Lecture name"] = topNames
    outDF["Lecture score"] = lectureScores
    outDF["Timestep list (score)"] = topTimeStamps
    return outDF

print("Gradient descent approach:")
print(tfidf_search("gradient descent approach", 2, 2))
print("\nBeer and diapers:")
print(tfidf_search("beer and diapers", 2, 2))

Gradient descent approach:
           Lecture name  Lecture score  \
0         04-regression       0.075626   
1  06-neural-networks-1       0.030053   

                               Timestep list (score)  
0  [[3324.14, 3353.17] (0.28494126693723254), [19...  
1  [[2272.58, 2301.78] (0.44955830639910144), [19...  

Beer and diapers:
           Lecture name  Lecture score  \
0  10-frequent-itemsets       0.028497   
1       01-introduction       0.002959   

                               Timestep list (score)  
0  [[1521.46, 1549.01] (0.4118660500855288), [857...  
1  [[3881.79, 3904.34] (0.22997246273557603), [2....  
